# 06 — Candidate Site Ranking

Extracts, ranks, and maps top candidate sites for both healthcare facilities and agricultural development zones.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import rasterio
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from utils import load_config, setup_logger
from site_ranking import extract_candidate_sites, rank_candidate_sites

logger = setup_logger("06_site_ranking")
hc_cfg = load_config(ROOT / "config/healthcare_config.yml")
ag_cfg = load_config(ROOT / "config/agriculture_config.yml")

OUT_R = ROOT / "outputs/rasters"
OUT_V = ROOT / "outputs/vectors"
OUT_M = ROOT / "outputs/maps"
INTERIM = ROOT / "data/interim"


## Healthcare — Extract Candidate Sites

In [ ]:
hc_suit = OUT_R / "healthcare_suitability.tif"
hc_rank_cfg = hc_cfg["site_ranking"]
study_area_path = INTERIM / "study_area.gpkg"

if hc_suit.exists():
    study_area = gpd.read_file(study_area_path) if study_area_path.exists() else None
    hc_sites = extract_candidate_sites(
        suitability_path=hc_suit,
        min_score=hc_rank_cfg["min_suitability_score"],
        min_area_ha=hc_rank_cfg["min_patch_area_ha"],
        study_area_gdf=study_area,
        output_path=OUT_V / "healthcare_candidate_sites_raw.gpkg",
    )
    print(f"Extracted {len(hc_sites)} raw candidate sites")
    if len(hc_sites):
        print(hc_sites[["mean_suitability","max_suitability","area_ha"]].describe().round(3))
else:
    print("Healthcare suitability raster not found. Run notebook 03 first.")
    hc_sites = gpd.GeoDataFrame()


## Healthcare — Rank & Export Top Sites

In [ ]:
if len(hc_sites):
    roads_path = ROOT / "data/raw/roads/roads_osm.gpkg"
    roads_gdf = gpd.read_file(roads_path) if roads_path.exists() else None

    hc_ranked = rank_candidate_sites(
        sites_gdf=hc_sites,
        ranking_criteria=hc_rank_cfg["ranking_criteria"],
        uncertainty_path=OUT_R / "healthcare_uncertainty_cv.tif",
        roads_gdf=roads_gdf,
        top_n=hc_rank_cfg["top_n_sites"],
    )
    hc_ranked.to_file(OUT_V / "healthcare_candidate_sites.gpkg", driver="GPKG")
    print(f"\nTop {len(hc_ranked)} healthcare candidate sites:")
    display_cols = ["rank","mean_suitability","area_ha","distance_to_road","uncertainty_score","composite_score"]
    display_cols = [c for c in display_cols if c in hc_ranked.columns]
    print(hc_ranked[display_cols].to_string(index=False))


## Agriculture — Extract & Rank Candidate Sites

In [ ]:
ag_suit = OUT_R / "agriculture_suitability.tif"
ag_rank_cfg = ag_cfg["site_ranking"]

if ag_suit.exists():
    study_area = gpd.read_file(study_area_path) if study_area_path.exists() else None
    ag_sites = extract_candidate_sites(
        suitability_path=ag_suit,
        min_score=ag_rank_cfg["min_suitability_score"],
        min_area_ha=ag_rank_cfg["min_patch_area_ha"],
        study_area_gdf=study_area,
        output_path=OUT_V / "agriculture_candidate_sites_raw.gpkg",
    )
    print(f"Extracted {len(ag_sites)} raw candidate sites")

    if len(ag_sites):
        ag_ranked = rank_candidate_sites(
            sites_gdf=ag_sites,
            ranking_criteria=ag_rank_cfg["ranking_criteria"],
            uncertainty_path=OUT_R / "agriculture_uncertainty_cv.tif",
            roads_gdf=roads_gdf if roads_path.exists() else None,
            top_n=ag_rank_cfg["top_n_sites"],
        )
        ag_ranked.to_file(OUT_V / "agriculture_candidate_sites.gpkg", driver="GPKG")
        print(f"\nTop {len(ag_ranked)} agriculture candidate sites saved.")
else:
    print("Agriculture suitability raster not found. Run notebook 04 first.")
    ag_ranked = gpd.GeoDataFrame()


## Side-by-Side Map — Top Sites

In [ ]:
def plot_ranked_sites(sites_gdf, suit_path, title, color, cmap_suit, ax):
    if not suit_path.exists() or len(sites_gdf) == 0:
        ax.text(0.5, 0.5, "Data not yet available",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontsize=10)
        return
    with rasterio.open(suit_path) as src:
        arr = src.read(1)
        extent = [src.bounds.left, src.bounds.right,
                  src.bounds.bottom, src.bounds.top]
    arr_m = np.where(arr <= 0, np.nan, arr)
    ax.imshow(arr_m, cmap=cmap_suit, vmin=0, vmax=1,
              extent=extent, origin="upper", alpha=0.85)

    sites_proj = sites_gdf.to_crs(rasterio.open(suit_path).crs)
    sites_proj.plot(ax=ax, color=color, edgecolor="white",
                    linewidth=0.8, alpha=0.85)

    # Label top 5
    for _, row in sites_proj.head(5).iterrows():
        c = row.geometry.centroid
        ax.annotate(str(row["rank"]), xy=(c.x, c.y),
                    fontsize=7, ha="center", va="center",
                    color="white", fontweight="bold")
    ax.set_title(title, fontsize=11)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

hc_sites_plot = hc_ranked if "hc_ranked" in dir() and len(hc_ranked) else gpd.GeoDataFrame()
ag_sites_plot = ag_ranked if "ag_ranked" in dir() and len(ag_ranked) else gpd.GeoDataFrame()

plot_ranked_sites(hc_sites_plot, OUT_R / "healthcare_suitability.tif",
                  "Healthcare — Top Candidate Sites", "#c00000", "RdYlGn", axes[0])
plot_ranked_sites(ag_sites_plot, OUT_R / "agriculture_suitability.tif",
                  "Agriculture — Top Candidate Sites", "#002060", "YlGn", axes[1])

plt.suptitle("Ranked Candidate Sites — Rural LGAs, Oyo State", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(OUT_M / "candidate_sites_map.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_M / 'candidate_sites_map.png'}")


## Summary Statistics

In [ ]:
print("=" * 55)
print("CANDIDATE SITE SUMMARY")
print("=" * 55)

for label, ranked in [("Healthcare", hc_ranked if "hc_ranked" in dir() else gpd.GeoDataFrame()),
                       ("Agriculture", ag_ranked if "ag_ranked" in dir() else gpd.GeoDataFrame())]:
    if len(ranked) == 0:
        continue
    print(f"\n{label} ({len(ranked)} sites):")
    print(f"  Mean suitability score : {ranked['mean_suitability'].mean():.3f}")
    print(f"  Mean area (ha)         : {ranked['area_ha'].mean():.1f}")
    if "composite_score" in ranked.columns:
        print(f"  Top site score         : {ranked.iloc[0]['composite_score']:.3f}")
        print(f"  Top site area (ha)     : {ranked.iloc[0]['area_ha']:.1f}")

print("\nOutputs saved:")
for f in sorted(OUT_V.glob("*.gpkg")):
    print(f"  {f.name}")
